# Graph structures

A unique feature to Graph Neural Networks is that these models learn by the enforcement of a spatial structure; a graph. When applied to epidemiological problems, GNNs should have access to meaningful graph structures which encode the (spatial) relations between the different entities modelled. A graph consists of **nodes** (entities) connected by **edges** (representing relationships between them). Edges may be weighted to quantify connection strength. In epidemiological applications, nodes may represent geographic regions such as districts or municipalities, while edges encode relationships relevant to disease transmission—including travel patterns or spatial proximity. In this project, a graph structure $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ represents Germany, with $N = |\mathcal{V}|$ nodes representing the districts and edges $\mathcal{E}$ encoding various forms of spatial connectivity. GNNs employ message-passing mechanisms that enable each node to aggregate information from its connected neighbors, effectively capturing the relational dynamics within the network. This ability to integrate diverse types of information across graph structures makes GNNs particularly promising for epidemiological modeling (Kraemer, 2025)


### Graph Structures used
In this project, multiple graph construction strategies are explored, to capture different aspects of spatial connectivity. Per administrative unit in Germany, albeit *NUTS1*, *NUTS2* or *NUTS3*, we model epidemiological timeseries, representing incidence rates on casenumbers of the respective infectious disease. The following classes of graph structures are studied:

- **Identity graph**:  A baseline graph structure in which each node is only connected to itself. Therefore, for the prediction of the next state of node $i$, only information of its own past state is used.
- **Mesh graph**: A baseline graph structure in which each node is connected to all other nodes by equal weight.
- **Boolean Neighbors**: A graph representing geographical information by connecting every node to the nodes it shares a geographic border with.
- **Gravity-model**: Graphs encoding geographical and socio-demographic information based on the gravity model of spatial interaction. For any pair of nodes $i, j$, edge weights are computed analogously to the gravitational force between two objects. The connection strength is a function of the population size of $i, j$, and the inverse of their distance. Two variations are used, one with $k=3$ (gravity1) and one with $k=7$ (gravity2).
- **Commuter-based**: Graphs representing commuting data made available by the Bundesagentur für Arbeit \cite{Arbeitsagentur}. Two variations are used based on the commuting data for 2024, one with $k=3$ (commuter 1) and one in which each node's connections are kept, so long as there are more than 1 000 daily commuters between them (commuter2).

 In addition, a threshold of $k$ connections per node may be implemented, emphasizing strong connections over many connections.


In [ ]:
from src.utils import get_data_env
from src.dataloading import EpiConfig, DataOrchestrator

disease_name    = 'influenza'
nuts_level      = 'nuts3'
min_date        = '2006-05-15'
max_date        = '2020-06-01'
split_trainval  = '2018-06-01'
split_valtest   = '2019-06-01'
split_berlin    = False

horizon_size    = 1
horizon_leadtime= 3
sequence_length = 1
lag_num         = 1


In [ ]:
config = EpiConfig(
    disease             = 'influenza',
    data_env_dir        = get_data_env(),
    date_range          = (min_date, max_date),
    horizon_size        = horizon_size,
    sequence_length     = sequence_length,
    horizon_leadtime    = horizon_leadtime,
    lag_num             = lag_num,
    nuts_level          = nuts_level,
    log_transform       = ['incidence'],
    split_berlin        = False,
    include_population  = False,
    split_trainval      = split_trainval, 
    split_valtest       = split_valtest,
    target_column       = 'cases',
    lag_column          = 'incidence',  
    verbose             = 0
    )    
data_orchestrator = DataOrchestrator(config).build()

In [ ]:
# import os
# import torch
# import numpy as np
# import seaborn as sns
# from dataclasses import dataclass, asdict
# from typing import Optional, List, Union, Literal, Dict, Tuple

# from src.dataloading.graphconstruction.edgeweight_normalizer import EdgeWeightNormalizer
# from src.dataloading.graphconstruction.graphconstructor import GraphConstructor
# from src.dataloading.graphconstruction.selfloop_adder import SelfLoopAdder
# from src.dataloading.graphconstruction.graphviewer import GraphViewer

# from src.dataloading.dataorchestration.dataorchestrator import DataOrchestrator

# from src.utils.textformatting import checkmark, warning_emoji, align

# @dataclass 
# class GraphStructure:
#     """
#     Single graphstructure with

#     Parameters
#     ----------
#     edge_index: torch.Tensor
#         index of edges
#         shape [num_edges, 2]
#     edge_weight: torch.Tensor
#         weights of edges
#         shape [num_edges, 1]
#     """
#     edge_index:     torch.Tensor 
#     edge_weight:    torch.Tensor    
    
#     def __repr__(self) -> str:
#         num_nodes      = len(self.edge_index[0].unique())
#         num_edges      = len(self.edge_index)
#         representation = f'<GraphStructure(num_nodes = {num_nodes}, num_edges = {num_edges})>'
#         return representation

# @dataclass 
# class GraphStatistics:
#     """
#     Contains statistics describing a graph
    
#     Parameters
#     ----------
#     #### General
#     num_nodes: int

#     num_edges: int

#     edge_density: float

#     num_isolated_nodes: int

#     #### Edge weights

#     edge_weight_mean: float

#     edge_weight_min: float

#     edge_weight_max: float

#     #### Out-degree

#     out_degree_mean:    float

#     out_degree_max:     int

#     out_degree_min:     int
    
#     #### In-degree
#     in_degree_mean:    float

#     in_degree_max:     int

#     in_degree_min:     int    
#     """
    
#     num_nodes:          int
#     num_edges:          int
#     edge_density:       float
#     num_isolated_nodes: int

#     edge_weight_mean:   float
#     edge_weight_min:    float
#     edge_weight_max:    float

#     out_degree_mean:    Union[int,float]
#     out_degree_max:     int
#     out_degree_min:     int    
   
#     in_degree_mean:     Union[int,float]
#     in_degree_max:      int
#     in_degree_min:      int    

#     def __repr__(self) -> str:

#         largest_key = len("num_isolated_nodes")

#         statement = self._get_small_summary() +"\n"
                        
#         statement += align('edge_weight_mean',   self.edge_weight_mean,  width=largest_key + 2, newline=True)   
#         statement += align('edge_weight_min',    self.edge_weight_min,   width=largest_key + 2, newline=True)                   
#         statement += align('edge_weight_max',    self.edge_weight_max,   width=largest_key + 2, newline=True)    

#         statement += "\n"      
#         statement += align('out_degree_mean',   self.out_degree_mean,  width=largest_key + 2, newline=True)   
#         statement += align('out_degree_max',    self.out_degree_max,   width=largest_key + 2, newline=True)                   
#         statement += align('out_degree_min',    self.out_degree_min,   width=largest_key + 2, newline=True)      

#         statement += "\n"      
#         statement += align('in_degree_mean',   self.in_degree_mean,  width=largest_key + 2, newline=True)   
#         statement += align('in_degree_max',    self.in_degree_max,   width=largest_key + 2, newline=True)                   
#         statement += align('in_degree_min',    self.in_degree_min,   width=largest_key + 2, newline=True)                

#         return statement      

#     def _get_small_summary(self) -> str:

#         largest_key = len("num_isolated_nodes")

#         statement = ""

#         statement += align('num_nodes',          self.num_nodes,            width=largest_key + 2, newline=True)
#         statement += align('num_edges',          self.num_edges,            width=largest_key + 2, newline=True)
#         statement += align('edge_density',       self.edge_density,         width=largest_key + 2, newline=True)
#         statement += align('num_isolated_nodes', self.num_isolated_nodes,   width=largest_key + 2, newline=True)                                 

#         return statement           

# @dataclass 
# class GraphEntry:
#     """
#     Single entry to the GraphRegistry with 

#     Parameters
#     ---------
#     structure:  GraphStructure

#     summary:
    
#     config:
#     """

#     structure: GraphStructure
#     summary:   GraphStatistics
#     config:    Dict[str,str]

#     def __repr__(self) -> str:
#         representation = f'<GraphEntry(structure, summary, config)>'
#         return representation        

#     def _get_summary(self, type: Literal['small','large']) -> str:
        
#         if type == 'large':
#             return print(self.summary)

#         elif type == 'small':
#             return print(self.summary._get_small_summary())


#     # def get_summary(self) -> str:
#     #     pass

# class GraphRegistry:
#     """
#     Registry of GraphStructure objects

#     Attributes
#     ----------
#     registry: Dict[str, GraphEntry]
#         graphnames : GraphEntry object
    
#     Methods
#     -------
#     add_entry       ->  None
#     rename_entry    ->  None
#     get_entry       ->  GraphEntry
#     """
#     def __init__(self):
#         self.registry: Dict[str, GraphEntry] = {}
#         self.alignment_width = 19

#     def add_entry(self, graphname: str, entry: GraphEntry) -> None:
#         """Add structure to .registry under graphname"""
#         if self.check_entry(graphname):
#             print(align(f'{warning_emoji} warning', f'{graphname} already exists, please rename the already existing entry. New entry wasn\'t registered', width=self.alignment_width, newline=False))            
#         else:
#             self.registry[graphname] = entry
#             print(align(f'{checkmark} Graph registered', f'{graphname} successfully registered', width=self.alignment_width, newline=False))                        
        
#     def rename_entry(self, current_graphname: str, new_graphname: str) -> None:
#         """rename entry from current_graphname to new_graphname; current_graphname is removed"""        
#         if self.check_entry(new_graphname):
#             print(align(f'{warning_emoji} warning', f'{new_graphname} already exists, please rename the already existing entry. New entry wasn\'t registered', width=self.alignment_width, newline=False))    
#         else:
#             self.add_entry(new_graphname,self.registry[current_graphname])
#             self.remove_entry(current_graphname)           

#     def remove_entry(self, graphname: str) -> None:
#         del self.registry[graphname]
#         print(align(f'{checkmark} Graph removed', f'{graphname} has been deregistered', width=self.alignment_width, newline=False))        

#     def get_entry(self, graphname: str) -> 'GraphEntry':
#         """return GraphStructure from graphname"""
#         if not self.check_entry(graphname):
#             print(f'{warning_emoji} {graphname} not found')
#             registered_entries = ', '.join(self.return_entrynames())
#             raise ValueError(f"the following graphs are registered:\n{registered_entries}")
        
#         else:
#             return self.registry[graphname]

#     def check_entry(self, graphname: str) -> bool:
#         """return boolean reflecting whether or not graphname is registered"""
#         if graphname in self.registry.keys():
#             return True
#         else:
#             return False
        
#     def return_entrynames(self) -> List[str]:
#         """returns a list of entrynames"""
#         return list(self.registry.keys())
        
#     def get_graph_stats(self, graphname: str, type: Literal['small','large']) -> str:
#         """Returns a string representation of the graph statistics, either small or extensive"""
        
#         return self.get_entry(graphname)._get_summary(type)

#     def __repr__(self) -> str:
#         registered_entries = ', '.join(self.return_entrynames())
#         return f'<GraphRegistry({registered_entries})'

# @dataclass
# class GraphConfig:
#     """
#     Config with which graph structure was created
#     """
#     method:         str
#     name_addition:  Optional[str]
#     self_connection:str
#     scaling_method: Optional[str]
#     kwargs:         dict

# palette_blues = sns.color_palette("Blues", n_colors=100)
# palette_reds  = sns.color_palette("Reds", n_colors=100)


# class GraphOrchestrator:
#     """
#     Orchestrates the entire process of graph creation

#     Parameters
#     ----------
#     data_orchestrator: DataOrchestrator
#         object with which data orchestration was created
#     id_col: str = 'node'
#         column used accross dataframes to refer to nodes
#     graph_dir: str = 'data/graphs'
#         directory in which to store, and from which to retrieve, graphs

#     Methods
#     -------
#     generate_graph  
#     preview_graph


#     Examples
#     --------
#     >>> orch                = data_orchestrator
#     >>> graphconstruction   = GraphOrchestrator(data_orchestrator=data_orchestrator)
#     >>> graphconstruction.generate_graph('commuter', scaling_method='rowwise')
#     >>> graphconstruction.rename_graph('commuter_selfmean_rowwise', 'commuter')
#     >>> graphconstruction.generate_graph('boolean_neighbors')
#     >>> graphconstruction.rename_graph('boolean_neighbors_selfmean', 'boolean_neighbors')

#     >>> graphconstruction.preview_graph('commuter', node_idx = 223, subplots = True, title= "Preview commuter graph from Munich")
#     >>> graphconstruction.preview_graph('boolean_neighbors', subplots = True, title= "Preview boolean neighbors graph")

#     Limitations #TODO
#     -----------
#     - population_size is determined as average per node over all years
#     - currently deals with static graphs only, dynamic graphs should be dealt with
#     - no selfloops visualization
    
#     See Also
#     ------------
#     GraphRegistry -> registry of graph structures (.graph_registry)
#     GraphViewer   -> graph previewer object (.previewer)


#     """
#     def __init__(self,                  
#                  data_orchestrator: DataOrchestrator,
#                  id_col:          str = 'node',
#                  graph_dir:       str = "data/graphs/"):
        
#         # extract metadata
#         shapes               = data_orchestrator.data_context.shapedata
#         self.tokens          = data_orchestrator.data_context.tokenization_map['id_idx']
#         self.epipopdata      = data_orchestrator.data_harmonized.data
        
#         # mean population data TODO => currently mean is taken instead of yearly
#         self.population_data = self.epipopdata.groupby(id_col)['population_size'].mean().reset_index()
#         self.shapes          = shapes.copy()
#         self.id_col          = id_col 
#         self.nuts_level      = data_orchestrator.data_context.nuts_level
#         self.graph_dir       = os.path.join(graph_dir, f'{self.nuts_level}')

#         # registry of graphs
#         self.graph_registry = GraphRegistry()
#         self.previewer      = GraphViewer(self.graph_registry, self.shapes)
#         os.makedirs(self.graph_dir, exist_ok=True)

#         self.graph_methods          = ['boolean_neighbors', 
#                                        'identity', 
#                                        'mesh', 
#                                        'distance_threshold',
#                                        'k_nearest', 
#                                        'population_weighted', 
#                                        'gravity_model', 
#                                        'commuter']
        
#         self.num_nodes              = data_orchestrator.data_context.num_nodes
        
#     def generate_graph(self, 
#                        method: str                                          = 'boolean_neighbors',
#                        name_addition:   Optional[str]                       = None,
#                        self_connection:  Literal['max','0','mean']          = 'mean',
#                        scaling_method:  Optional[Literal['minmax','log','zscore','symmetric','rowwise']] = None,
#                        **kwargs) -> None:
#         """
#         Generates a graph structure based on the method. Depending on the method, additional kwargs may be required.
#         A graph structure and config are created and saved into the dictionary `self.graph_registry` under the key
#         corresponding to `graph_name`, which is equal to:

#             method + name_addition + self{self_connection} + scaling_method

#         where '_' is used as separator

#         Parameters
#         ----------
#         method: str 
#             limited to: ['boolean_neighbors', 'identity', 'mesh', 'distance_threshold','k_nearest',  'population_weighted', 'gravity_model', 'commuter']
#         name_addition: Optional[str]
#             adjustment of graphname
#         self_connection: Literal['max','0','mean']
#             how to deal with self_connection
#         scaling_method: Optional[Literal['minmax','log','zscore','symmetric','rowwise']] = None
#             how to deal with edge_weights
#         **kwargs:
#             kwargs are method-specific.

#         See also
#         --------
#         The heavy lifting is done through the following classes. Each of these contains further information.
#             - GraphGeneration
#             - GraphEdgeWeightNormalizer
#             - GraphAddSelfLoops
#         """

#         if method not in self.graph_methods:
#             raise ValueError(f'{method} not a valid graph method. Please choose a method from this list:\n{self.graph_methods}')

#         graphconfig = GraphConfig(
#             method          = method,
#             name_addition   = name_addition,
#             self_connection = self_connection,
#             scaling_method  = scaling_method,
#             kwargs          = kwargs
#             )

        
#         graphname = f'{method}_{name_addition}'         if name_addition    else f'{method}'
#         graphname = f'{graphname}_self{self_connection}'if self_connection  else f'{graphname}'
#         graphname = f'{graphname}_{scaling_method}'     if scaling_method   else f'{graphname}'

#         # Ensure IDs are integers and no missing
#         shapes_cp               = self.shapes.dropna(subset=[self.id_col])
#         shapes_cp[self.id_col]  = shapes_cp[self.id_col].astype(int)
#         node_ids                = np.array(shapes_cp[self.id_col].dropna().astype(int).values)

#         ##########################
#         ##### Create Graphs ######
#         ##########################     
#         graph_generator = GraphConstructor(
#             gdf     = shapes_cp,
#             tokens  = self.tokens,
#             popdata = self.population_data,
#             id_col  = self.id_col
#         )

#         # Generate the graph with whatever method and kwargs
#         edges, weights = graph_generator.generate_graph(method=method, **kwargs)

#         # if weights is undefined, give 1 everywhere
#         if weights is None:
#             weights = [float(1) for _ in edges]

#         ##########################
#         ##### Add self-loops #####
#         ##########################        
#         if method not in ['identity', 'mesh']:
#             edges, weights = SelfLoopAdder(edge_indices=edges, edge_weights=weights, node_ids=node_ids).add_loops(self_connection)

#         # remove zero valued loops        
#         edges, weights = self._remove_zero_weights(edges, weights)

#         # transform into torch objects        
#         edge_weight = torch.tensor(weights, dtype=torch.float)
#         edge_index  = torch.tensor(edges, dtype=torch.long).t().contiguous()

#         ##########################
#         # Normalize edge-weights #
#         ##########################
#         if scaling_method:
#            edge_weight = EdgeWeightNormalizer(edge_indices=edge_index, edge_weights=edge_weight, num_nodes = self.num_nodes).normalize(scaling_method)

#         edge_index, edge_weight = self._remove_zero_weights(edge_index, edge_weight)

#         graphstructure = GraphStructure(edge_index, edge_weight)
#         graphentry     = GraphEntry(graphstructure, self._generate_graph_stats(graphstructure), asdict(graphconfig))

#         # save config
#         # graphdict = {'structure': graphstructure,
#         #              'config'   : asdict(graphconfig),
#         #              'summary'  : self._get_graph_summary(edge_index,edge_weight)}   

#         self.graph_registry.add_entry(graphname, graphentry)
#         # print(f'{checkmark} graph generated: {graphname}')

#     def _remove_zero_weights(
#         self, 
#         edge_index: Union[List[Tuple[int, int]], torch.Tensor], 
#         edge_weight: Union[List[float], torch.Tensor], 
#         threshold: float = 1e-9
#     ) -> Tuple[Union[List[Tuple[int, int]], torch.Tensor], Union[List[float], torch.Tensor]]:
#         """
#         Remove edges with zero or near-zero weights.
        
#         Parameters
#         ----------
#         edge_index : Union[List[Tuple[int, int]], torch.Tensor]
#             Edge indices as list of tuples or tensor [2, num_edges]
#         edge_weight : Union[List[float], torch.Tensor]
#             Edge weights as list or tensor [num_edges]
#         threshold : float
#             Values below this are considered zero (default: 1e-9)
            
#         Returns
#         -------
#         Tuple containing filtered edge_index and edge_weight
#         """
#         if isinstance(edge_index, torch.Tensor):
#             if not isinstance(edge_weight, torch.Tensor):
#                 raise TypeError(
#                     f'edge_index is torch.Tensor but edge_weight is {type(edge_weight).__name__}'
#                 )
            
#             mask = edge_weight.abs() > threshold
#             return edge_index[:, mask], edge_weight[mask]
        
#         elif isinstance(edge_index, list):
#             if not isinstance(edge_weight, list):
#                 raise TypeError(
#                     f'edge_index is list but edge_weight is {type(edge_weight).__name__}'
#                 )
            
#             # Filter and unzip in one go
#             filtered = [(e, w) for e, w in zip(edge_index, edge_weight) if abs(w) > threshold]
            
#             if not filtered:  # Handle empty case
#                 return [], []
            
#             filtered_edges, filtered_weights = zip(*filtered)
#             return list(filtered_edges), list(filtered_weights)
        
#         else:
#             raise TypeError(
#                 f'edge_index must be list or torch.Tensor, got {type(edge_index).__name__}'
#             )       

#     def preview_graph(self, graphname: str, node_idx: Optional[int] = None, subplots: bool = True, title: Optional[str] = None):        
#         """
#         Preview graph found in registry

#         Parameters
#         ----------
#         graphname: str
#             the name under which the graph structure is saved in the registry (.graph_registry shows registered graphs)
#             for viewing an empty graphstructure (unconnected nodes) use graphname = 'empty'
#         node_idx: Optional[int] = None
#             the node of which to view the neighborhood (when int)
#             by default, view global graph (node_idx = None)
#         subplots: bool = True
#             whether or not to show more (distributions) than just a global map
#         title: Optional[str] = None
#             title for the main (global) map
            
#         See Also
#         --------
#         GraphViewer -> does the actual heavy lifting. This method simply relays parameters.
#         """
#         return self.previewer.view(graphname, node_idx, subplots, title)

#     def rename_graph(self, old_graphname: str, new_graphname: str) -> None:
#         """ 
#         Rename a graph in the registry (the key by which the graph is saved)
#         the old graph is copied into the `new graphname` and the `old_graphname` is removed.
#         """
#         self.graph_registry.rename_entry(old_graphname, new_graphname)
        
#     def get_graph_stats(self, graphname: str, type: Optional[Literal['small','large']] = 'large') -> str:
#         """Returns a string representation of the graph statistics, either small or extensive"""
#         self.graph_registry.get_graph_stats(graphname, type)

#     def save_graph(self, graphname: Union[str,List[str]] = 'all') -> None:
#         """
#         Save edge index and weight from registry. 
#         If graphname == 'all', all graphs are saved.
#         """

#         if graphname == ['all']:
#             graphname = 'all'

#         if graphname == 'all':
#             graph_entries_to_save = self.graph_registry.return_entrynames()

#         elif isinstance(graphname, str):
#             graph_entries_to_save = [graphname]

#         else:
#             raise ValueError(f'Please provide a list or string for the graphname.')

#         for graphname in graph_entries_to_save:

#             graph_entry = self.graph_registry.get_entry(graphname)
#             if graph_entry is not None:
                
#                 edge_index  = graph_entry.edge_index
#                 edge_weight = graph_entry.edge_weight

#                 torch.save(edge_index, os.path.join(self.graph_dir, f'{graphname}_edge_index.pt'))
#                 print(f'{checkmark} graph saved: edge index {graphname} saved to {self.graph_dir}')

#                 if edge_weight is not None:
#                     torch.save(edge_weight, os.path.join(self.graph_dir, f'{graphname}_edge_weight.pt'))
#                     print(f'{checkmark} graph saved: edge weight {graphname} saved to {self.graph_dir}')

#     def _generate_graph_stats(self, graph_structure: GraphStructure) -> GraphStatistics:
#         """ 
#         Returns a summary of the graph
#         """
#         global_edge_index = graph_structure.edge_index
#         global_edge_weight = graph_structure.edge_weight

#         # statistics
#         num_edges = global_edge_index.shape[1]
#         num_nodes = int(global_edge_index.max().item()) + 1
#         edge_density = num_edges / (num_nodes * (num_nodes - 1))
#         edge_weight_np = global_edge_weight.cpu().numpy()
        
#         # Round edge weight statistics at creation time
#         edge_weight_mean = round(float(edge_weight_np.mean()), 4)
#         edge_weight_min = round(float(edge_weight_np.min()), 4)
#         edge_weight_max = round(float(edge_weight_np.max()), 4)

#         # isolated nodes:
#         edges_out, edges_in = global_edge_index[0], global_edge_index[1] 
#         out_degree = torch.bincount(edges_out, minlength=num_nodes)
#         in_degree = torch.bincount(edges_in, minlength=num_nodes)
#         isolated_mask = (out_degree == 0) & (in_degree == 0)
#         num_isolated_nodes = isolated_mask.sum().item()

#         # Out-degree stats
#         out_degree_mean = round(float(out_degree.float().mean().item()), 2)
#         out_degree_max = out_degree.max().item()
#         out_degree_min = out_degree[out_degree > 0].min().item() if (out_degree > 0).any() else 0

#         # In-degree stats
#         in_degree_mean = round(float(in_degree.float().mean().item()), 2)
#         in_degree_max = in_degree.max().item()
#         in_degree_min = in_degree[in_degree > 0].min().item() if (in_degree > 0).any() else 0

#         return GraphStatistics(
#             num_edges=num_edges,
#             num_nodes=num_nodes,
#             edge_density=round(edge_density, 4),
#             edge_weight_mean=edge_weight_mean,
#             edge_weight_min=edge_weight_min,
#             edge_weight_max=edge_weight_max,
#             num_isolated_nodes=num_isolated_nodes,
#             out_degree_mean=out_degree_mean,
#             out_degree_max=out_degree_max,
#             out_degree_min=out_degree_min,
#             in_degree_mean=in_degree_mean,
#             in_degree_max=in_degree_max,
#             in_degree_min=in_degree_min
#         )
    
#     def __repr__(self) -> str:
#         return f'<GraphConstructor(level {self.nuts_level}. Registry: {self.graph_registry})>'

In [ ]:
# from src.dataloading.graphconstruction import GraphOrchestrator

orch                = data_orchestrator
graphconstruction   = GraphOrchestrator(data_orchestrator=data_orchestrator)
figure_empty        = graphconstruction.preview_graph('empty', title= "Preview Germany NUTS3")

# identity graph
graphconstruction.generate_graph(method = 'identity')
graphconstruction.rename_graph('identity_selfmean', 'identity_graph')
figure_identity_graph = graphconstruction.preview_graph('identity_graph', node_idx = 26, subplots = True, title= "Preview identity_graph for Hannover")

# mesh graph
graphconstruction.generate_graph(method = 'mesh')
graphconstruction.rename_graph('mesh_selfmean', 'mesh_graph')
figure_mesh_graph = graphconstruction.preview_graph('mesh_graph', node_idx = 26, subplots = True, title= "Preview mesh_graph for Hannover")

# boolean neighbors
graphconstruction.generate_graph(method='boolean_neighbors')
graphconstruction.rename_graph('boolean_neighbors_selfmean','boolean_neighbors_self')
figure_boolean_neighbors_self = graphconstruction.preview_graph('boolean_neighbors_self', node_idx = 26, subplots = True, title= "Preview boolean_neighbors_self for Hannover")

graphconstruction.generate_graph(method='boolean_neighbors', self_connection='0')
graphconstruction.rename_graph('boolean_neighbors_self0','boolean_neighbors_nonself')
figure_boolean_neighbors_nonself = graphconstruction.preview_graph('boolean_neighbors_nonself', node_idx = 26, subplots = True, title= "Preview boolean_neighbors_nonself for Hannover")

graphconstruction.generate_graph(method='boolean_neighbors', scaling_method='rowwise')
graphconstruction.rename_graph('boolean_neighbors_selfmean_rowwise','neighbors_self_rw')
figure_neighbors_selfrw = graphconstruction.preview_graph('neighbors_self_rw', node_idx = 26, subplots = True, title= "Preview neighbors self rw for Hannover")

graphconstruction.generate_graph(method='boolean_neighbors', scaling_method='rowwise', self_connection='0')
graphconstruction.rename_graph('boolean_neighbors_self0_rowwise','neighbors_nonself_rw')
figure_neighbors_nonselfrw = graphconstruction.preview_graph('neighbors_nonself_rw', node_idx = 26, subplots = True, title= "Preview neighbors nonself rw for Hannover")


# sparse - long distance gravity model
graphconstruction.generate_graph(method             = 'gravity_model', 
                                 self_connection    = '0',
                                 max_distance       = 1000_000,
                                 top_k              = 3,
                                 alpha              = 1,
                                 decay              = 1,
                                 scaling_method     = 'rowwise'
                                 )
graphconstruction.rename_graph('gravity_model_self0_rowwise','gravity1')
figure_gravity1 = graphconstruction.preview_graph('gravity1', node_idx = 26, subplots = True, title= "Preview gravity1 for Hannover")

# sparse - long distance gravity model
graphconstruction.generate_graph(method             = 'gravity_model', 
                                 self_connection    = '0',
                                 max_distance       = 1000_000,
                                 top_k              = 10,
                                 alpha              = 1,
                                 decay              = 1,
                                 scaling_method     = 'rowwise'
                                 )
graphconstruction.rename_graph('gravity_model_self0_rowwise','gravity2')
figure_gravity2 = graphconstruction.preview_graph('gravity2', node_idx = 26, subplots = True, title= "Preview gravity2 for Hannover")

# sparse - short distance gravity model
graphconstruction.generate_graph(method             = 'gravity_model', 
                                 self_connection    = '0',
                                 max_distance       = 1000_000,
                                 top_k              = 3,
                                 alpha              = 2,
                                 decay              = 1,
                                 scaling_method     = 'rowwise'
                                 )
graphconstruction.rename_graph('gravity_model_self0_rowwise','gravity3')
figure_gravity3 = graphconstruction.preview_graph('gravity3', node_idx = 26, subplots = True, title= "Preview gravity3 for Hannover")

# dense - short distance gravity model
graphconstruction.generate_graph(method             = 'gravity_model', 
                                 self_connection    = '0',
                                 max_distance       = 1000_000,
                                 top_k              = 10,
                                 alpha              = 2,
                                 decay              = 1,
                                 scaling_method     = 'rowwise'
                                 )
graphconstruction.rename_graph('gravity_model_self0_rowwise','gravity4')
figure_gravity4 = graphconstruction.preview_graph('gravity4', node_idx = 26, subplots = True, title= "Preview gravity4 for Hannover")

# dense - medium distance gravity model
graphconstruction.generate_graph(method             = 'gravity_model', 
                                 self_connection    = '0',
                                 max_distance       = 1000_000,
                                 top_k              = 7,
                                 alpha              = 1.5,
                                 decay              = 1,
                                 scaling_method     = 'rowwise'
                                 )
graphconstruction.rename_graph('gravity_model_self0_rowwise','gravity5')
figure_gravity5 = graphconstruction.preview_graph('gravity5', node_idx = 26, subplots = True, title= "Preview gravity5 for Hannover")

    ✓ Graph registered  : identity_selfmean successfully registered
    ✓ Graph registered  : identity_graph successfully registered
    ✓ Graph removed     : identity_selfmean has been deregistered
    ✓ Graph registered  : mesh_selfmean successfully registered
    ✓ Graph registered  : mesh_graph successfully registered
    ✓ Graph removed     : mesh_selfmean has been deregistered
    ✓ Graph registered  : boolean_neighbors_selfmean successfully registered
    ✓ Graph registered  : boolean_neighbors_self successfully registered
    ✓ Graph removed     : boolean_neighbors_selfmean has been deregistered
    ✓ Graph registered  : boolean_neighbors_self0 successfully registered
    ✓ Graph registered  : boolean_neighbors_nonself successfully registered
    ✓ Graph removed     : boolean_neighbors_self0 has been deregistered
    ✓ Graph registered  : boolean_neighbors_selfmean_rowwise successfully registered
    ✓ Graph registered  : neighbors_self_rw successfully registered
    ✓ Graph re

In [ ]:


graphconstruction.generate_graph(
    method='commuter', 
    self_connection='0',
    commuter_type = 'static',
    commuting_threshold = 1000,
    top_k = 4,
    scaling_method='rowwise',
    name_addition='1'
)

graphconstruction.generate_graph(
    method='commuter', 
    self_connection='0',
    commuter_type = 'static',
    commuting_threshold = 1000,
    top_k = 4,
    scaling_method='rowwise',
    name_addition='2'
)

graphconstruction.generate_graph(
    method='commuter', 
    self_connection='0',
    commuter_type = 'static',
    commuting_threshold = 1000,
    scaling_method='rowwise',
    name_addition='3'
)

graphconstruction.generate_graph(
    method='commuter', 
    self_connection='0',
    commuter_type = 'static',
    commuting_threshold = 1000,
    scaling_method='rowwise',
    name_addition='4'
)

graphconstruction.rename_graph('identity_self0',                'identity_graph')
graphconstruction.rename_graph('boolean_neighbors_selfmax',     'boolean_neighbors')
graphconstruction.rename_graph('boolean_neighbors_self0',       'boolean_neighbors_nonself')
graphconstruction.rename_graph('gravity_model_self0_rowwise',   'gravity_graph')
graphconstruction.rename_graph('gravity_model_2_self0_rowwise', 'gravity2')
graphconstruction.rename_graph('gravity_model_3_self0_rowwise', 'gravity3')
graphconstruction.rename_graph('gravity_model_4_self0_rowwise', 'gravity4')
graphconstruction.rename_graph('commuter_1_self0_rowwise',      'commuter1')
graphconstruction.rename_graph('commuter_2_self0_rowwise',   'commuter2')
graphconstruction.rename_graph('commuter_3_self0_rowwise',      'commuter_graph')
graphconstruction.rename_graph('commuter_4_self0_rowwise',   'commuter4')



# graphconstruction.save_graph('all')